# Apollonian-Style Packing Inside a Shape

This notebook adapts the spirit of `matlab/apolonian/Apolonian.m` using a standalone Python implementation.

We iteratively insert the largest empty $\ell_p$ ball into a binary domain.

## Optimization viewpoint

Let $\Omega\subset\mathbb{R}^2$ be a shape and $B_t$ the set of balls already inserted.

At iteration $t$, we choose center $c_t\in\Omega\setminus B_t$ maximizing distance to the current occupied boundary:

$$
c_t = \arg\max_{x\in\Omega\setminus B_t} \min_{y\in\partial(\Omega\setminus B_t)} d_p(x,y).
$$

Then we add the ball

$$
\{x : d_p(x,c_t) < r_t\},\quad r_t = \min_{y\in\partial(\Omega\setminus B_t)} d_p(c_t,y).
$$

Different $p$ produce very different visual packings.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from ipywidgets import interact, IntSlider, Dropdown

plt.rcParams['figure.dpi'] = 120
rng = np.random.default_rng(11)

## 1) Build a binary domain

In [ ]:
def make_domain(n=280, kind='star'):
    y, x = np.mgrid[-1:1:complex(0, n), -1:1:complex(0, n)]
    r = np.sqrt(x**2 + y**2)
    th = np.arctan2(y, x)

    if kind == 'star':
        rho = 0.58 + 0.12 * np.cos(5 * th) + 0.06 * np.sin(3 * th)
        mask = r < rho
        mask &= (x > -0.95) & (x < 0.95) & (y > -0.95) & (y < 0.95)
    elif kind == 'bean':
        mask = ((x + 0.2)**2 / 0.55**2 + (y + 0.1)**2 / 0.43**2 < 1)
        mask |= ((x - 0.18)**2 / 0.33**2 + (y - 0.05)**2 / 0.28**2 < 1)
    else:  # ring
        mask = (r < 0.75) & (r > 0.32)
    return mask.astype(bool)

shape = make_domain(kind='star')

fig, ax = plt.subplots(figsize=(5.4, 5.4))
ax.imshow(shape, cmap='gray_r', origin='lower')
ax.set_title('Binary target domain')
ax.axis('off')
plt.show()

## 2) Boundary extraction and $\ell_p$ distances

In [ ]:
def extract_boundary(mask):
    # 4-neighborhood boundary (inside pixels touching outside)
    ero = ndimage.binary_erosion(mask)
    bnd = mask & (~ero)
    return np.argwhere(bnd)[:, ::-1]  # (x, y)

def lp_dist(A, B, p='2'):
    # A: (m,2), B: (k,2) -> distances (m,k)
    dx = np.abs(A[:, None, 0] - B[None, :, 0])
    dy = np.abs(A[:, None, 1] - B[None, :, 1])
    if p == 'inf':
        return np.maximum(dx, dy)
    pval = float(p)
    return (dx**pval + dy**pval) ** (1.0 / pval)

bnd = extract_boundary(shape)
inside = np.argwhere(shape)[:, ::-1]
print('Boundary points:', len(bnd), '| Interior points:', len(inside))

## 3) Greedy insertion of largest empty balls

In [ ]:
def draw_lp_ball(mask, center, radius, p='2'):
    h, w = mask.shape
    y, x = np.mgrid[0:h, 0:w]
    xy = np.stack([x.ravel(), y.ravel()], axis=1)
    c = np.array(center, dtype=float)[None, :]
    d = lp_dist(xy, c, p=p).reshape(h, w)
    out = mask.copy()
    out[d < radius] = False
    return out, (d < radius)

def run_packing(domain, p='2', n_iter=35, n_inside=2200, n_bnd=2400, seed=0):
    rg = np.random.default_rng(seed)
    W = domain.copy()  # True means free interior still available
    H, Wd = domain.shape
    canvas = np.ones((H, Wd, 3), dtype=float)
    colors = plt.cm.turbo(np.linspace(0.02, 0.95, n_iter))[:, :3]
    history = [canvas.copy()]
    balls = []

    for it in range(n_iter):
        inside = np.argwhere(W)[:, ::-1]
        if len(inside) < 40:
            break
        bnd = extract_boundary(W)
        if len(bnd) < 10:
            break

        I = inside[rg.choice(len(inside), size=min(n_inside, len(inside)), replace=False)]
        B = bnd[rg.choice(len(bnd), size=min(n_bnd, len(bnd)), replace=False)]

        D = lp_dist(I.astype(float), B.astype(float), p=p)
        dmin = D.min(axis=1)
        k = int(np.argmax(dmin))
        c = I[k]
        r = float(dmin[k])

        W, disk = draw_lp_ball(W, c, r, p=p)
        color = colors[it % len(colors)]
        canvas[disk] = color
        balls.append((c, r))
        history.append(canvas.copy())
    return history, balls

frames, balls = run_packing(shape, p='2', n_iter=36, seed=9)
print('Balls inserted:', len(balls))

## 4) Interactive rendering

In [ ]:
def show_frame(t=0):
    t = min(t, len(frames) - 1)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(frames[t], origin='lower')
    ax.set_title(f'Apollonian-style packing, step {t}/{len(frames)-1}')
    ax.axis('off')
    plt.show()

interact(show_frame, t=IntSlider(min=0, max=len(frames)-1, step=1, value=len(frames)-1));

## 5) Compare $p$-norm geometries

In [ ]:
def preview_norm(p='2', n_iter=24):
    hist, _ = run_packing(shape, p=p, n_iter=n_iter, seed=5)
    fig, ax = plt.subplots(figsize=(5.8, 5.8))
    ax.imshow(hist[-1], origin='lower')
    ax.set_title(f'Final packing with p={p}')
    ax.axis('off')
    plt.show()

interact(preview_norm, p=Dropdown(options=['1', '2', 'inf', '0.6'], value='2'), n_iter=IntSlider(min=10, max=40, step=2, value=24));

## 6) Takeaways

- The process greedily approximates a maximal empty-ball decomposition.
- Sampling gives a fast approximation of the exact farthest-point query.
- Geometry depends strongly on the metric ($\ell_1$, $\ell_2$, $\ell_\infty$, or quasi-norms).